In [1]:
# =============================================================================
# MSCI FACTOR GARCH/EGARCH ANALYSIS — yfinance Live Data
# Broader timeframe: 2014-01-01 → today
# Packages: pip install yfinance arch scipy statsmodels pandas numpy matplotlib seaborn
# =============================================================================
# !pip install yfinance pandas numpy scipy statsmodels matplotlib seaborn arch --- IGNORE ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
from scipy import stats, optimize
from statsmodels.stats.diagnostic import acorr_ljungbox
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# SECTION 1: DOWNLOAD LIVE MSCI FACTOR ETF DATA (yfinance)
# =============================================================================
# best available free proxies for each MSCI factor:
TICKERS = {
    "SPX":      "SPY",   # S&P 500 — benchmark
    "Momentum": "MTUM",  # iShares MSCI USA Momentum Factor (inception Apr 2013)
    "Value":    "VLUE",  # iShares MSCI USA Value Factor    (inception Apr 2013)
    "Quality":  "QUAL",  # iShares MSCI USA Quality Factor  (inception Jul 2013)
    "Min_Vol":  "USMV",  # iShares MSCI USA Min Vol Factor  (inception Oct 2011)
    "Size":     "IWM",   # iShares Russell 2000 — size proxy (since 2000)
}

START_DATE = "2014-01-01"   # All 6 ETFs fully active from here
END_DATE   = "2026-02-21"   # Current date

print("=== DOWNLOADING MSCI FACTOR ETF DATA ===")
raw = yf.download(
    list(TICKERS.values()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,   # already adjusted for dividends/splits
    progress=False
)

# Extract Adj Close (auto_adjust=True gives 'Close' as adjusted)
prices = raw["Close"].copy()
prices.columns = list(TICKERS.keys())   # rename SPY→SPX etc.
prices = prices.dropna()

print(f"Data: {prices.index[0].date()} → {prices.index[-1].date()} | {len(prices)} trading days")
print(prices.tail(3))

# Daily log returns (%)
returns = np.log(prices / prices.shift(1)).dropna() * 100   # in % for ARCH
cum_returns = (returns / 100 + 1).cumprod()

factor_cols = list(TICKERS.keys())

=== DOWNLOADING MSCI FACTOR ETF DATA ===
Data: 2014-01-02 → 2026-02-20 | 3052 trading days
                   SPX    Momentum       Value     Quality    Min_Vol  \
Date                                                                    
2026-02-18  263.989990  253.490005  203.490005  686.289978  96.029999   
2026-02-19  264.600006  253.300003  203.029999  684.479980  96.050003   
2026-02-20  264.609985  254.779999  204.389999  689.429993  96.190002   

                  Size  
Date                    
2026-02-18  152.300003  
2026-02-19  151.380005  
2026-02-20  152.419998  


In [3]:
# =============================================================================
# SECTION 2: DESCRIPTIVE STATISTICS
# =============================================================================
print("\n=== DESCRIPTIVE STATISTICS ===")
desc = returns.describe().T
desc["skewness"] = returns.skew()
desc["kurtosis"] = returns.kurtosis()
print(desc.round(4))
desc.to_csv("descriptive_stats.csv")


=== DESCRIPTIVE STATISTICS ===
           count    mean     std      min     25%     50%     75%      max  \
SPX       3051.0  0.0327  1.3928 -14.2335 -0.6816  0.0913  0.7994   8.7545   
Momentum  3051.0  0.0524  1.2559 -13.1992 -0.4899  0.1134  0.6741  10.1169   
Value     3051.0  0.0482  1.1001 -10.7379 -0.3855  0.0717  0.5764   9.8612   
Quality   3051.0  0.0502  1.0910 -11.5887 -0.3638  0.0659  0.5724   9.9863   
Min_Vol   3051.0  0.0403  0.8873 -10.6269 -0.3087  0.0750  0.4558   8.3124   
Size      3051.0  0.0407  1.1874 -13.5681 -0.4871  0.0434  0.6437   9.5589   

          skewness  kurtosis  
SPX        -0.6924    8.9587  
Momentum   -0.5291   10.8483  
Value      -0.4033   12.8278  
Quality    -0.5891   14.9585  
Min_Vol    -0.8958   21.6309  
Size       -0.8656   16.2426  


In [4]:
# =============================================================================
# SECTION 3: PLOTS — Price, Cumulative Returns, Correlations
# =============================================================================
def save_plot(fig, name):
    fig.savefig(name, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {name}")

# Prices
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), factor_cols):
    prices[col].plot(ax=ax, lw=0.8, color="steelblue")
    ax.set_title(f"{col} ({TICKERS[col]})", fontsize=10)
    ax.set_ylabel("Price (USD)")
fig.suptitle("MSCI factor ETF prices (2014–2026)", fontsize=13)
plt.tight_layout()
save_plot(fig, "01_prices.png")

# Cumulative returns
fig, ax = plt.subplots(figsize=(12, 5))
for col in factor_cols:
    cum_returns[col].plot(ax=ax, lw=1.5, label=col)
ax.set_title("Cumulative returns by factor (2014–2026)", fontsize=12)
ax.set_ylabel("Cumulative return (rebased to 1)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
save_plot(fig, "02_cum_returns.png")

# Full-period correlation
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(returns.corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Factor return correlations (2014–2026)", fontsize=11)
plt.tight_layout()
save_plot(fig, "03_corr_full.png")

  Saved 01_prices.png
  Saved 02_cum_returns.png
  Saved 03_corr_full.png


In [5]:
# =============================================================================
# SECTION 4: BROADER PERIOD SUBSAMPLES (extends R COVID subsamples)
# =============================================================================
broad_periods = {
    "Pre-GFC recovery":  ("2014-01-01", "2016-12-31"),
    "Late bull market":  ("2017-01-01", "2019-12-31"),
    "COVID crisis":      ("2020-01-01", "2020-12-31"),
    "Recovery rebound":  ("2021-01-01", "2021-12-31"),
    "Rate hike regime":  ("2022-01-01", "2023-12-31"),
    "Post hike":         ("2024-01-01", "2026-02-21"),
}

# Narrow COVID sub-periods (from original R)
covid_periods = {
    "Incubation":  ("2020-01-01", "2020-02-21"),
    "Fever":       ("2020-02-22", "2020-03-20"),
    "Treatment":   ("2020-03-21", "2020-04-15"),
    "Recovery":    ("2020-04-16", "2020-12-31"),
}

all_periods = {**broad_periods, **covid_periods}
subsamples = {}

for name, (s, e) in all_periods.items():
    mask = (returns.index >= s) & (returns.index <= e)
    sub = returns.loc[mask].dropna()
    subsamples[name] = sub
    print(f"  {name}: {len(sub)} obs")

# Per-period correlation heatmaps (2x5 grid)
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for ax, (name, sub) in zip(axes.flatten(), all_periods.items()):
    sns.heatmap(subsamples[name].corr(), annot=True, fmt=".1f",
                    cmap="RdBu_r", center=0, ax=ax, cbar=False, annot_kws={"size": 7})
    ax.set_title(name.replace("_", " "), fontsize=8)
plt.suptitle("Cross-period factor correlations", fontsize=12)
plt.tight_layout()
save_plot(fig, "04_period_correlations.png")

  Pre-GFC recovery: 755 obs
  Late bull market: 754 obs
  COVID crisis: 253 obs
  Recovery rebound: 252 obs
  Rate hike regime: 501 obs
  Post hike: 536 obs
  Incubation: 35 obs
  Fever: 20 obs
  Treatment: 17 obs
  Recovery: 181 obs
  Saved 04_period_correlations.png


In [6]:
# =============================================================================
# SECTION 5: DIAGNOSTIC HELPER FUNCTIONS
# =============================================================================
def run_jb(residuals):
    """Jarque-Bera normality test using scipy.stats."""
    result = stats.jarque_bera(residuals)
    return float(result.statistic), float(result.pvalue)

def run_ljungbox(residuals, lag=10):
    """Ljung-Box autocorrelation test using statsmodels."""
    lb = acorr_ljungbox(residuals, lags=[lag], return_df=True)
    return float(lb["lb_pvalue"].iloc[0])

In [7]:
# =============================================================================
# SECTION 6: MASTER FITTING FUNCTION — GARCH(1,1) vs EGARCH(1,1)
# =============================================================================
def fit_garch_egarch(ret_series: pd.Series, name: str) -> dict:
    """
    Fit GARCH(1,1) and EGARCH(1,1) on a % return series.
    Returns dict with params, vol, AIC, persistence, diagnostics.
    """
    y = ret_series.dropna()     # already in % from Section 1

    # ---- GARCH(1,1) --------------------------------------------------------
    garch = arch_model(y, vol="Garch", p=1, q=1, dist="normal", mean="Zero")
    g_fit = garch.fit(disp="off")
    g_vol = g_fit.conditional_volatility / 100.0    # back to decimal

    alpha1_g = float(g_fit.params.get("alpha[1]", 0.0))
    beta1_g  = float(g_fit.params.get("beta[1]",  0.0))
    g_persist = alpha1_g + beta1_g

    # ---- EGARCH(1,1) -------------------------------------------------------
    egarch = arch_model(y, vol="EGarch", p=1, o=1, q=1, dist="normal", mean="Zero")
    e_fit = egarch.fit(disp="off")
    e_vol = e_fit.conditional_volatility / 100.0

    alpha1_e = float(e_fit.params.get("alpha[1]", 0.0))
    beta1_e  = float(e_fit.params.get("beta[1]",  0.0))
    gamma    = float(e_fit.params.get("gamma[1]", np.nan))
    e_persist = alpha1_e + beta1_e

    # ---- Diagnostics on GARCH standardized residuals -----------------------
    std_resid = g_fit.std_resid.dropna().values
    jb_stat, jb_p = run_jb(std_resid)
    lb_resid_p  = run_ljungbox(std_resid,    lag=10)
    lb_sq_p     = run_ljungbox(std_resid**2, lag=10)

    return {
        "factor":       name,
        "g_vol":        g_vol,
        "e_vol":        e_vol,
        "g_aic":        g_fit.aic,
        "e_aic":        e_fit.aic,
        "g_bic":        g_fit.bic,
        "e_bic":        e_fit.bic,
        "g_persist":    g_persist,
        "e_persist":    e_persist,
        "e_gamma":      gamma,
        "aic_diff":     e_fit.aic - g_fit.aic,
        "jb_stat":      jb_stat,
        "jb_p":         jb_p,
        "lb_resid_p":   lb_resid_p,
        "lb_sq_p":      lb_sq_p,
        "g_fit":        g_fit,
        "e_fit":        e_fit,
    }

In [8]:
# =============================================================================
# SECTION 7: FIT ALL FACTORS
# =============================================================================
print("\n=== FITTING GARCH/EGARCH TO ALL FACTORS ===")
all_results = {}
for col in factor_cols:
    print(f"  {col} ...", end=" ")
    all_results[col] = fit_garch_egarch(returns[col], col)
    print(f"GARCH AIC {all_results[col]['g_aic']:.1f} | "
          f"EGARCH AIC {all_results[col]['e_aic']:.1f} | "
          f"gamma {all_results[col]['e_gamma']:.4f}")


=== FITTING GARCH/EGARCH TO ALL FACTORS ===
  SPX ... GARCH AIC 9860.2 | EGARCH AIC 9773.7 | gamma -0.0997
  Momentum ... GARCH AIC 8820.3 | EGARCH AIC 8713.2 | gamma -0.1364
  Value ... GARCH AIC 7850.6 | EGARCH AIC 7671.7 | gamma -0.1804
  Quality ... GARCH AIC 7767.0 | EGARCH AIC 7610.7 | gamma -0.1807
  Min_Vol ... GARCH AIC 6335.3 | EGARCH AIC 6226.7 | gamma -0.1605
  Size ... GARCH AIC 8415.9 | EGARCH AIC 8323.6 | gamma -0.1315


In [9]:
# =============================================================================
# SECTION 8: IN-SAMPLE COMPARISON TABLE
# =============================================================================
rows = []
for col, res in all_results.items():
    rows.append({
        "Factor":           col,
        "Ticker":           TICKERS[col],
        "GARCH_AIC":        round(res["g_aic"], 2),
        "GARCH_Persist":    round(res["g_persist"], 4),
        "EGARCH_AIC":       round(res["e_aic"], 2),
        "EGARCH_Gamma":     round(res["e_gamma"], 4),
        "AIC_Diff":         round(res["aic_diff"], 2),
        "Better_Model":     "EGARCH" if res["aic_diff"] < 0 else "GARCH",
        "JB_Stat":          round(res["jb_stat"], 2),
        "JB_Pval":          f"{res['jb_p']:.2e}",
        "LB_Resid_P":       round(res["lb_resid_p"], 4),
        "LB_SqResid_P":     round(res["lb_sq_p"], 4),
    })

comp_df = pd.DataFrame(rows)
comp_df.to_csv("CompleteAnalysisComparison.csv", index=False)
print("\n=== IN-SAMPLE COMPARISON TABLE ===")
print(comp_df.to_string(index=False))



=== IN-SAMPLE COMPARISON TABLE ===
  Factor Ticker  GARCH_AIC  GARCH_Persist  EGARCH_AIC  EGARCH_Gamma  AIC_Diff Better_Model  JB_Stat   JB_Pval  LB_Resid_P  LB_SqResid_P
     SPX    SPY    9860.24         0.9764     9773.69       -0.0997    -86.55       EGARCH   271.06  1.38e-59      0.8923        0.6847
Momentum   MTUM    8820.29         0.9760     8713.20       -0.1364   -107.09       EGARCH   587.09 3.27e-128      0.6750        0.2533
   Value   VLUE    7850.58         0.9768     7671.72       -0.1804   -178.86       EGARCH   841.73 1.66e-183      0.7863        0.1136
 Quality   QUAL    7766.99         0.9641     7610.72       -0.1807   -156.27       EGARCH  1033.31 4.16e-225      0.7503        0.0959
 Min_Vol   USMV    6335.26         0.9640     6226.68       -0.1605   -108.57       EGARCH  1595.51  0.00e+00      0.3759        0.6075
    Size    IWM    8415.92         0.9662     8323.64       -0.1315    -92.28       EGARCH   376.68  1.60e-82      0.4029        0.1159


In [10]:
# =============================================================================
# SECTION 9: VOLATILITY OVERLAY PLOT (all 6 factors)
# =============================================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, col in zip(axes.flatten(), factor_cols):
    res = all_results[col]
    idx = returns.index
    n   = len(res["g_vol"])
    ax.fill_between(idx[-n:], returns[col].iloc[-n:] / 100,
                    alpha=0.25, color="gray", label="Returns")
    ax.plot(idx[-n:], res["g_vol"], lw=1.0, color="steelblue", label="GARCH(1,1)")
    ax.plot(idx[-n:], res["e_vol"], lw=1.0, color="crimson",
            linestyle="--", label="EGARCH(1,1)")
    ax.axvline(pd.Timestamp("2020-03-16"), color="orange",
               lw=0.8, linestyle=":", label="COVID Crash")
    ax.axvline(pd.Timestamp("2022-01-01"), color="purple",
               lw=0.8, linestyle=":", label="Rate Hikes")
    ax.set_title(f"{col} ({TICKERS[col]}) | γ={res['e_gamma']:.3f}", fontsize=10)
    ax.set_ylabel("Volatility (σ)")
axes[0][0].legend(fontsize=7, loc="upper left")
fig.suptitle("Conditional volatility: GARCH(1,1) vs EGARCH(1,1) over full sample (2014–2026)", fontsize=12)
plt.tight_layout()
save_plot(fig, "05_vol_overlay.png")

  Saved 05_vol_overlay.png


In [11]:
# =============================================================================
# SECTIONS 10–14: ELEVATED ANALYSIS (builds on all_results + returns from prev script)
# NO re-fitting, NO rolling OOS — uses conditional vol already computed
# =============================================================================
# Requires: all_results, returns, factor_cols, TICKERS, prices (from Section 1–9)
# Install:  pip install scipy statsmodels numpy pandas matplotlib seaborn

# =============================================================================
# SECTION 10: DYNAMIC VaR BACKTEST + KUPIEC PROPORTION OF FAILURES TEST
# Concept: Does GARCH vs EGARCH conditional vol correctly calibrate downside risk?
# =============================================================================
def save_plot(fig, fname):
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {fname}")

def kupiec_pof(violations: np.ndarray, alpha: float) -> tuple:
    """
    Kupiec (1995) POF test.
    H0: actual failure rate = alpha (model is correctly calibrated).
    Returns LR statistic and p-value (chi-squared, df=1).
    """
    T = len(violations)
    N = violations.sum()
    if N == 0 or N == T:
        return np.nan, np.nan
    p = N / T
    LR = -2 * (N * np.log(alpha / p) + (T - N) * np.log((1 - alpha) / (1 - p)))
    p_val = 1 - stats.chi2.cdf(LR, df=1)
    return round(LR, 4), round(p_val, 4)


print("\n=== SECTION 10: DYNAMIC VaR BACKTEST ===")

alphas = [0.01, 0.05]          # 99% VaR and 95% VaR
var_rows = []

for col in factor_cols:
    res = all_results[col]
    # Align returns (%) and conditional vol (decimal)
    ret_pct   = returns[col].values                  # in %
    g_vol_pct = res["g_vol"].values * 100            # decimal → %
    e_vol_pct = res["e_vol"].values * 100

    # Make sure lengths match (conditional vol may be same length as returns)
    n = min(len(ret_pct), len(g_vol_pct))
    ret_pct   = ret_pct[-n:]
    g_vol_pct = g_vol_pct[-n:]
    e_vol_pct = e_vol_pct[-n:]

    for alpha in alphas:
        z = stats.norm.ppf(alpha)              # negative critical value

        g_var = z * g_vol_pct                  # VaR in %
        e_var = z * e_vol_pct

        g_viols = (ret_pct < g_var).astype(int)
        e_viols = (ret_pct < e_var).astype(int)

        g_lr, g_p = kupiec_pof(g_viols, alpha)
        e_lr, e_p = kupiec_pof(e_viols, alpha)

        # CVaR = Expected Shortfall (mean loss conditional on breach)
        g_cvar = ret_pct[g_viols.astype(bool)].mean() if g_viols.sum() > 0 else np.nan
        e_cvar = ret_pct[e_viols.astype(bool)].mean() if e_viols.sum() > 0 else np.nan

        var_rows.append({
            "Factor":           col,
            "VaR_Level":        f"{int((1-alpha)*100)}%",
            "Expected_Breaches": f"{alpha*100:.0f}%",
            "GARCH_Breaches_%": round(g_viols.mean() * 100, 2),
            "EGARCH_Breaches_%":round(e_viols.mean() * 100, 2),
            "GARCH_Kupiec_P":   g_p,
            "EGARCH_Kupiec_P":  e_p,
            "GARCH_CVaR_%":     round(g_cvar, 4),
            "EGARCH_CVaR_%":    round(e_cvar, 4),
        })

var_df = pd.DataFrame(var_rows)
var_df.to_csv("VaR_Backtest_Kupiec.csv", index=False)
print(var_df.to_string(index=False))

# — VaR violation chart for SPX (both models, 99%)
col = "SPX"
res = all_results[col]
n = min(len(returns[col]), len(res["g_vol"]))
idx_plot     = returns.index[-n:]
ret_plot     = returns[col].values[-n:]
g_var_plot   = stats.norm.ppf(0.01) * res["g_vol"].values[-n:] * 100
e_var_plot   = stats.norm.ppf(0.01) * res["e_vol"].values[-n:] * 100
g_viols_plot = ret_plot < g_var_plot
e_viols_plot = ret_plot < e_var_plot

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, var_line, viols, model, color in [
    (axes[0], g_var_plot, g_viols_plot, "GARCH(1,1)", "steelblue"),
    (axes[1], e_var_plot, e_viols_plot, "EGARCH(1,1)", "crimson"),
]:
    ax.bar(idx_plot, ret_plot, width=1, color="lightgray", label="Daily return (%)")
    ax.plot(idx_plot, var_line, lw=1.2, color=color, label=f"{model} 99% VaR")
    ax.scatter(idx_plot[viols], ret_plot[viols], color="red", s=10, zorder=5,
               label=f"Violations: {viols.sum()} ({viols.mean()*100:.2f}%)")
    ax.axvline(pd.Timestamp("2020-03-16"), color="orange", lw=0.8,
               linestyle=":", alpha=0.8, label="COVID crash")
    ax.axvline(pd.Timestamp("2022-01-01"), color="purple", lw=0.8,
               linestyle=":", alpha=0.8, label="Rate hike start")
    ax.legend(fontsize=8, loc="lower left")
    ax.set_title(f"SPX — {model} 99% VaR | Kupiec p = "
                 f"{[r['GARCH_Kupiec_P'] if model=='GARCH(1,1)' else r['EGARCH_Kupiec_P'] for r in var_rows if r['Factor']=='SPX' and r['VaR_Level']=='99%'][0]}")
    ax.set_ylabel("Return (%)")

fig.suptitle("Dynamic VaR backtest: GARCH vs EGARCH (SPX, 2014–2026)", fontsize=12)
plt.tight_layout()
save_plot(fig, "10_var_backtest_spx.png")


=== SECTION 10: DYNAMIC VaR BACKTEST ===
  Factor VaR_Level Expected_Breaches  GARCH_Breaches_%  EGARCH_Breaches_%  GARCH_Kupiec_P  EGARCH_Kupiec_P  GARCH_CVaR_%  EGARCH_CVaR_%
     SPX       99%                1%              1.77               1.41          0.0001           0.0323       -3.8263        -4.1384
     SPX       95%                5%              4.72               5.01          0.4736           0.9702       -2.9758        -2.9296
Momentum       99%                1%              2.10               1.97          0.0000           0.0000       -3.3101        -3.2109
Momentum       95%                5%              5.80               5.83          0.0474           0.0392       -2.5633        -2.5940
   Value       99%                1%              2.03               2.00          0.0000           0.0000       -2.7723        -2.7786
   Value       95%                5%              4.72               5.11          0.4736           0.7752       -2.3356        -2.2364
 Quali

In [12]:
# =============================================================================
# SECTION 11: NEWS IMPACT CURVES (NIC)
# Concept: Visualize how each model responds to positive vs negative return shocks
# GARCH: symmetric U-shape | EGARCH: asymmetric — negative shock → larger vol
# =============================================================================
print("\n=== SECTION 11: NEWS IMPACT CURVES ===")

def compute_nic(g_fit, e_fit, sigma_bar: float, n_pts: int = 300):
    """
    Compute NIC for GARCH and EGARCH over a range of shocks.
    sigma_bar: unconditional volatility (used as base level).
    Returns shock grid (ε) and variance responses for both models.
    """
    # Extract GARCH params
    omega_g = float(g_fit.params["omega"])
    alpha_g = float(g_fit.params["alpha[1]"])
    beta_g  = float(g_fit.params["beta[1]"])
    sigma2_bar = sigma_bar ** 2

    # Extract EGARCH params
    omega_e = float(e_fit.params["omega"])
    alpha_e = float(e_fit.params["alpha[1]"])
    gamma_e = float(e_fit.params["gamma[1]"])
    beta_e  = float(e_fit.params["beta[1]"])

    # Shock grid: -4σ to +4σ
    eps = np.linspace(-4 * sigma_bar, 4 * sigma_bar, n_pts)
    z   = eps / sigma_bar                             # standardized shock

    # GARCH NIC: σ²_t = ω + α·ε² + β·σ̄²  (σ̄² held at unconditional)
    nic_g = omega_g + alpha_g * eps**2 + beta_g * sigma2_bar

    # EGARCH NIC: ln(σ²_t) = ω/(1−β) + α·|z| + γ·z
    # (β·ln(σ̄²) absorbed into constant at unconditional level)
    ln_sigma2_base = omega_e / (1 - beta_e)
    ln_nic_e       = ln_sigma2_base + alpha_e * np.abs(z) + gamma_e * z
    nic_e          = np.exp(ln_nic_e)

    return eps, nic_g, nic_e


fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, col in zip(axes, factor_cols):
    res   = all_results[col]
    g_fit = res["g_fit"]
    e_fit = res["e_fit"]

    # Unconditional vol from GARCH (in % — same scale as fitted model)
    omega_g    = float(g_fit.params["omega"])
    alpha_g    = float(g_fit.params["alpha[1]"])
    beta_g     = float(g_fit.params["beta[1]"])
    unc_var    = omega_g / (1 - alpha_g - beta_g) if (alpha_g + beta_g) < 1 else omega_g / 0.01
    sigma_bar  = np.sqrt(unc_var)                 # in % scale

    eps, nic_g, nic_e = compute_nic(g_fit, e_fit, sigma_bar)

    # Normalise to 1 at ε=0 for visual comparison
    nic_g_norm = nic_g / nic_g[len(nic_g) // 2]
    nic_e_norm = nic_e / nic_e[len(nic_e) // 2]

    ax.plot(eps / sigma_bar, nic_g_norm, lw=1.5, color="steelblue", label="GARCH (symmetric)")
    ax.plot(eps / sigma_bar, nic_e_norm, lw=1.5, color="crimson",
            linestyle="--", label="EGARCH (asymmetric)")
    ax.axvline(0, color="black", lw=0.6, linestyle=":")
    ax.set_title(f"{col} | γ={res['e_gamma']:+.4f}", fontsize=10)
    ax.set_xlabel("Shock (z = ε/σ̄)", fontsize=8)
    ax.set_ylabel("Norm. variance response", fontsize=8)
    ax.legend(fontsize=7)

fig.suptitle("News impact curves: GARCH vs EGARCH asymmetry across all factors",
             fontsize=12)
plt.tight_layout()
save_plot(fig, "11_news_impact_curves.png")



=== SECTION 11: NEWS IMPACT CURVES ===
  Saved 11_news_impact_curves.png


In [13]:
# =============================================================================
# SECTION 12: VOLATILITY REGIME ANALYSIS
# Concept: Classify each day as Low / Medium / High vol (GARCH conditional vol)
#          and analyse factor returns within each regime
# =============================================================================
print("\n=== SECTION 12: VOLATILITY REGIME ANALYSIS ===")

# Build a master DataFrame: returns + GARCH conditional vol per factor
n_min = min(len(returns), min(len(all_results[c]["g_vol"]) for c in factor_cols))
regime_df = returns[factor_cols].iloc[-n_min:].copy()

# Use SPX GARCH vol as the market-wide regime indicator (most natural)
spx_vol = pd.Series(
    all_results["SPX"]["g_vol"].values[-n_min:] * 100,  # back to %
    index=regime_df.index,
    name="SPX_Vol"
)
regime_df["SPX_Vol"] = spx_vol

# Classify: Low (≤ 33rd), Medium (33rd–67th), High (> 67th)
low_t  = spx_vol.quantile(0.33)
high_t = spx_vol.quantile(0.67)

def classify_vol(v):
    if v <= low_t:   return "Low Vol"
    if v <= high_t:  return "Medium Vol"
    return "High Vol"

regime_df["Regime"] = spx_vol.map(classify_vol)

# Summary stats per regime per factor
def regime_stats(df, regime_col, factor):
    grp = df.groupby(regime_col)[factor]
    return pd.DataFrame({
        "Mean_Return_%": grp.mean().round(4),
        "Std_%":         grp.std().round(4),
        "Sharpe_proxy":  (grp.mean() / grp.std()).round(4),
        "Hit_Rate_%":    (grp.apply(lambda x: (x > 0).mean()) * 100).round(2),
        "Max_Drawdown_%":(grp.apply(lambda x: x.min())).round(4),
    })

all_regime_stats = {}
for col in factor_cols:
    all_regime_stats[col] = regime_stats(regime_df, "Regime", col)

# Export
regime_summary = pd.concat(all_regime_stats, names=["Factor"])
regime_summary.to_csv("VolRegimeAnalysis.csv")
print(regime_summary)

# Heatmap: Sharpe proxy by factor × regime
sharpe_matrix = pd.DataFrame({
    col: all_regime_stats[col]["Sharpe_proxy"]
    for col in factor_cols
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(sharpe_matrix.T, annot=True, fmt=".3f",
            cmap="RdYlGn", center=0, ax=axes[0])
axes[0].set_title("Sharpe proxy by factor × Vol regime", fontsize=11)
axes[0].set_ylabel("Factor")

# Count of days per regime
regime_counts = regime_df["Regime"].value_counts()
regime_counts.plot(kind="bar", ax=axes[1], color=["green", "orange", "red"])
axes[1].set_title("Days per volatility regime", fontsize=11)
axes[1].set_ylabel("# Trading days")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
save_plot(fig, "12_vol_regime_analysis.png")

# Factor return distributions per regime
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, col in zip(axes.flatten(), factor_cols):
    for regime, color in [("Low Vol","green"),("Medium Vol","orange"),("High Vol","red")]:
        subset = regime_df[regime_df["Regime"] == regime][col]
        subset.plot.kde(ax=ax, label=regime, color=color, lw=1.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Daily return (%)")
    ax.legend(fontsize=7)
    ax.axvline(0, color="black", lw=0.5, linestyle=":")

fig.suptitle("Return distributions by volatility regime (SPX-based classification)", fontsize=12)
plt.tight_layout()
save_plot(fig, "12b_regime_return_distributions.png")


=== SECTION 12: VOLATILITY REGIME ANALYSIS ===
                     Mean_Return_%   Std_%  Sharpe_proxy  Hit_Rate_%  \
Factor   Regime                                                        
SPX      High Vol           0.0958  1.8635        0.0514       55.11   
         Low Vol           -0.0529  0.9230       -0.0573       51.44   
         Medium Vol         0.0545  1.2248        0.0445       52.84   
Momentum High Vol           0.1109  1.7353        0.0639       56.80   
         Low Vol            0.0129  0.7863        0.0164       55.71   
         Medium Vol         0.0339  1.0557        0.0321       53.42   
Value    High Vol           0.1033  1.5499        0.0667       54.62   
         Low Vol           -0.0043  0.6709       -0.0064       52.73   
         Medium Vol         0.0455  0.8872        0.0513       54.87   
Quality  High Vol           0.1059  1.5398        0.0688       56.31   
         Low Vol           -0.0071  0.6660       -0.0106       53.53   
         Medium 

In [14]:
# =============================================================================
# SECTION 13: VOLATILITY PERSISTENCE & HALF-LIFE RANKING
# Concept: Half-life = log(0.5) / log(α+β) — how fast vol reverts after a shock
# Key finding: high persistence → vol shocks last longer (crisis contagion proxy)
# =============================================================================
print("\n=== SECTION 13: VOLATILITY PERSISTENCE & HALF-LIFE ===")

halflife_rows = []
for col in factor_cols:
    res       = all_results[col]
    persist_g = res["g_persist"]
    persist_e = res["e_persist"]

    # Half-life in trading days (log(0.5)/log(persistence))
    hl_g = np.log(0.5) / np.log(persist_g) if 0 < persist_g < 1 else np.inf
    hl_e = np.log(0.5) / np.log(persist_e) if 0 < persist_e < 1 else np.inf

    halflife_rows.append({
        "Factor":          col,
        "Ticker":          TICKERS[col],
        "GARCH_Persist":   round(persist_g, 4),
        "EGARCH_Persist":  round(persist_e, 4),
        "GARCH_HalfLife_days":  round(hl_g, 1),
        "EGARCH_HalfLife_days": round(hl_e, 1),
        "EGARCH_Gamma":    round(res["e_gamma"], 4),
    })

hl_df = pd.DataFrame(halflife_rows).sort_values("GARCH_HalfLife_days", ascending=False)
hl_df.to_csv("Persistence_HalfLife.csv", index=False)
print(hl_df.to_string(index=False))

# Visualization: dual bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Persistence
hl_df.set_index("Factor")[["GARCH_Persist", "EGARCH_Persist"]].plot(
    kind="bar", ax=axes[0],
    color=["steelblue", "crimson"], alpha=0.85, edgecolor="white"
)
axes[0].set_title("Volatility persistence (α+β)", fontsize=11)
axes[0].set_ylabel("Persistence")
axes[0].axhline(1.0, color="black", lw=0.8, linestyle="--", label="Unit Root")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)

# Half-Life
hl_df.set_index("Factor")[["GARCH_HalfLife_days", "EGARCH_HalfLife_days"]].plot(
    kind="bar", ax=axes[1],
    color=["steelblue", "crimson"], alpha=0.85, edgecolor="white"
)
axes[1].set_title("Half-life of vol shock (trading days)", fontsize=11)
axes[1].set_ylabel("Days")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(fontsize=8)

# Leverage Effect (EGARCH gamma)
colors_gamma = ["crimson" if g < 0 else "green"
                for g in hl_df["EGARCH_Gamma"]]
axes[2].bar(hl_df["Factor"], hl_df["EGARCH_Gamma"], color=colors_gamma, alpha=0.8)
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_title("EGARCH leverage effect (γ)\nNegative = Bad news → Higher Vol", fontsize=10)
axes[2].set_ylabel("γ coefficient")
axes[2].tick_params(axis="x", rotation=45)

fig.suptitle("Volatility persistence, half-life & leverage effect — All Factors",
             fontsize=12)
plt.tight_layout()
save_plot(fig, "13_persistence_halflife_leverage.png")


=== SECTION 13: VOLATILITY PERSISTENCE & HALF-LIFE ===
  Factor Ticker  GARCH_Persist  EGARCH_Persist  GARCH_HalfLife_days  EGARCH_HalfLife_days  EGARCH_Gamma
   Value   VLUE         0.9768          1.1754                 29.5                   inf       -0.1804
     SPX    SPY         0.9764          1.1184                 29.0                   inf       -0.0997
Momentum   MTUM         0.9760          1.1770                 28.5                   inf       -0.1364
    Size    IWM         0.9662          1.1517                 20.1                   inf       -0.1315
 Quality   QUAL         0.9641          1.1743                 19.0                   inf       -0.1807
 Min_Vol   USMV         0.9640          1.1706                 18.9                   inf       -0.1605
  Saved 13_persistence_halflife_leverage.png


In [15]:
# =============================================================================
# SECTION 14: GARCH CONDITIONAL COVARIANCE PORTFOLIO OPTIMIZATION
# Concept: Build min-variance portfolio using current GARCH conditional vols
#          as diagonal + long-run correlation as off-diagonal
#          Compare: Equal-Weight vs Min-Variance vs Vol-Parity
# =============================================================================
print("\n=== SECTION 14: GARCH PORTFOLIO OPTIMIZATION ===")

# Current conditional vol vector (last observation, decimal)
current_vols = np.array([all_results[c]["g_vol"].values[-1] for c in factor_cols])

# Long-run correlation from full-period decimal returns
ret_dec      = returns[factor_cols] / 100
corr_matrix  = ret_dec.corr().values

# GARCH conditional covariance matrix = D * Corr * D
D     = np.diag(current_vols)
Sigma = D @ corr_matrix @ D

n_assets = len(factor_cols)

# Portfolio variance function
def port_vol(w, Sigma):
    return float(np.sqrt(w @ Sigma @ w))

# ---- Minimum Variance Portfolio (long-only) --------------------------------
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
bounds      = [(0.0, 1.0)] * n_assets
w0          = np.ones(n_assets) / n_assets

opt = optimize.minimize(
    fun=port_vol,
    x0=w0,
    args=(Sigma,),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"ftol": 1e-12, "maxiter": 1000}
)
w_minvar = opt.x

# ---- Vol Parity Portfolio (inverse-vol weighting) --------------------------
inv_vol  = 1.0 / current_vols
w_parity = inv_vol / inv_vol.sum()

# ---- Equal Weight -----------------------------------------------------------
w_equal = np.ones(n_assets) / n_assets

# Compare portfolio vols
pv_minvar = port_vol(w_minvar, Sigma) * np.sqrt(252) * 100
pv_parity = port_vol(w_parity, Sigma) * np.sqrt(252) * 100
pv_equal  = port_vol(w_equal,  Sigma) * np.sqrt(252) * 100

print(f"\nPortfolio Annualised Vol:")
print(f"  Equal Weight:   {pv_equal:.2f}%")
print(f"  Vol Parity:     {pv_parity:.2f}%")
print(f"  Min Variance:   {pv_minvar:.2f}%  ← GARCH-optimised")

# Weight table
weight_df = pd.DataFrame({
    "Factor":       factor_cols,
    "Ticker":       [TICKERS[c] for c in factor_cols],
    "GARCH_Vol_Ann%": (current_vols * np.sqrt(252) * 100).round(2),
    "EqualWeight_%":  (w_equal  * 100).round(2),
    "VolParity_%":    (w_parity * 100).round(2),
    "MinVar_%":       (w_minvar * 100).round(2),
})
weight_df.to_csv("GARCHPortfolioWeights.csv", index=False)
print(weight_df.to_string(index=False))

# ---- Simulated Efficient Frontier (random long-only portfolios) -------------
np.random.seed(42)
n_sim = 5000
sim_vols, sim_rets = [], []
ret_means = ret_dec[factor_cols].mean().values * 252   # annualised means

for _ in range(n_sim):
    w = np.random.dirichlet(np.ones(n_assets))         # random long-only weights
    pv = port_vol(w, Sigma) * np.sqrt(252) * 100
    pr = float(w @ ret_means) * 100
    sim_vols.append(pv)
    sim_rets.append(pr)

# Special portfolios
label_pts = {
    "Equal Weight":  (pv_equal,  float(w_equal @ ret_means) * 100, "blue",  "o"),
    "Vol Parity":    (pv_parity, float(w_parity @ ret_means) * 100, "orange","^"),
    "Min Variance":  (pv_minvar, float(w_minvar @ ret_means) * 100, "red",   "*"),
}
individual = {
    c: (current_vols[i] * np.sqrt(252) * 100,
        float(ret_means[i]) * 100)
    for i, c in enumerate(factor_cols)
}

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sim_vols, sim_rets, c=np.array(sim_rets) / np.array(sim_vols),
                cmap="RdYlGn", alpha=0.3, s=5)
plt.colorbar(sc, ax=ax, label="Sharpe proxy (Ret/Vol)")

for name, (pv, pr, color, marker) in label_pts.items():
    ax.scatter(pv, pr, color=color, marker=marker, s=200, zorder=5,
               edgecolors="black", linewidths=0.8, label=name)

for col_name, (pv, pr) in individual.items():
    ax.scatter(pv, pr, color="gray", marker="D", s=60, zorder=4)
    ax.annotate(col_name, (pv, pr), textcoords="offset points",
                xytext=(6, 4), fontsize=7, color="dimgray")

ax.set_xlabel("Annualised volatility (%)", fontsize=11)
ax.set_ylabel("Annualised return (%)", fontsize=11)
ax.set_title("GARCH-based efficient frontier\n(Current conditional covariance, "
             "5K simulated portfolios)", fontsize=12)
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
save_plot(fig, "14_garch_efficient_frontier.png")



=== SECTION 14: GARCH PORTFOLIO OPTIMIZATION ===

Portfolio Annualised Vol:
  Equal Weight:   14.28%
  Vol Parity:     13.26%
  Min Variance:   9.27%  ← GARCH-optimised
  Factor Ticker  GARCH_Vol_Ann%  EqualWeight_%  VolParity_%  MinVar_%
     SPX    SPY           19.42          16.67        12.09       0.0
Momentum   MTUM           20.22          16.67        11.62       0.0
   Value   VLUE           13.02          16.67        18.04       0.0
 Quality   QUAL           12.10          16.67        19.42       0.0
 Min_Vol   USMV            9.27          16.67        25.33     100.0
    Size    IWM           17.40          16.67        13.50       0.0
  Saved 14_garch_efficient_frontier.png


In [16]:
# =============================================================================
# FINAL SUMMARY TABLE: KEY FINDINGS ACROSS ALL SECTIONS
# =============================================================================
print("\n" + "="*65)
print("RESEARCH SUMMARY — GARCH-BASED MSCI FACTOR ANALYSIS (2014–2026)")
print("="*65)

summary = pd.DataFrame({
    "Factor":         factor_cols,
    "Ticker":         [TICKERS[c] for c in factor_cols],
    "Better_Model":   [("EGARCH" if all_results[c]["aic_diff"] < 0 else "GARCH")
                       for c in factor_cols],
    "Leverage_γ":     [round(all_results[c]["e_gamma"], 4) for c in factor_cols],
    "Vol_Persist_G":  [round(all_results[c]["g_persist"], 4) for c in factor_cols],
    "HalfLife_days":  hl_df.set_index("Factor")["GARCH_HalfLife_days"].reindex(factor_cols).values,
    "MinVar_Weight_%":(w_minvar * 100).round(1),
})
summary.to_csv("Research_Summary.csv", index=False)
print(summary.to_string(index=False))

print("\nOutputs:")
print("  CSVs: VaR_Backtest_Kupiec.csv | VolRegimeAnalysis.csv | "
      "Persistence_HalfLife.csv | GARCHPortfolioWeights.csv | Research_Summary.csv")
print("  PNGs: 10_var_backtest_spx.png | 11_news_impact_curves.png | "
      "12_vol_regime_analysis.png | 12b_regime_return_distributions.png | "
      "13_persistence_halflife_leverage.png | 14_garch_efficient_frontier.png")



RESEARCH SUMMARY — GARCH-BASED MSCI FACTOR ANALYSIS (2014–2026)
  Factor Ticker Better_Model  Leverage_γ  Vol_Persist_G  HalfLife_days  MinVar_Weight_%
     SPX    SPY       EGARCH     -0.0997         0.9764           29.0              0.0
Momentum   MTUM       EGARCH     -0.1364         0.9760           28.5              0.0
   Value   VLUE       EGARCH     -0.1804         0.9768           29.5              0.0
 Quality   QUAL       EGARCH     -0.1807         0.9641           19.0              0.0
 Min_Vol   USMV       EGARCH     -0.1605         0.9640           18.9            100.0
    Size    IWM       EGARCH     -0.1315         0.9662           20.1              0.0

Outputs:
  CSVs: VaR_Backtest_Kupiec.csv | VolRegimeAnalysis.csv | Persistence_HalfLife.csv | GARCHPortfolioWeights.csv | Research_Summary.csv
  PNGs: 10_var_backtest_spx.png | 11_news_impact_curves.png | 12_vol_regime_analysis.png | 12b_regime_return_distributions.png | 13_persistence_halflife_leverage.png | 14_garc

In [17]:
# # =============================================================================
# # SECTION 10: ROLLING OOS FORECASTS (SPX + Momentum)
# # =============================================================================
# def rolling_oos(series: pd.Series, window: int = 250, rv_days: int = 20):
#     """
#     Rolling one-step-ahead vol forecast.
#     - window: estimation window in days
#     - rv_days: realized vol window (std over next rv_days)
#     Returns metrics dict + forecast DataFrame.
#     """
#     y = series.dropna()       # already in %
#     n = len(y)
#     g_list, e_list, real_list, dates = [], [], [], []

#     for i in range(n - window - rv_days):
#         train = y.iloc[i : i + window]

#         # GARCH forecast
#         gm = arch_model(train, vol="Garch", p=1, q=1, dist="normal", mean="Zero")
#         gf = gm.fit(disp="off")
#         g_var = gf.forecast(horizon=1).variance.values[-1, 0]
#         g_list.append(np.sqrt(g_var) / 100.0)

#         # EGARCH forecast
#         em = arch_model(train, vol="EGarch", p=1, o=1, q=1, dist="normal", mean="Zero")
#         ef = em.fit(disp="off")
#         e_var = ef.forecast(horizon=1).variance.values[-1, 0]
#         e_list.append(np.sqrt(e_var) / 100.0)

#         # Realized vol (next rv_days)
#         rv = y.iloc[i + window : i + window + rv_days].std() / 100.0
#         real_list.append(rv)
#         dates.append(y.index[i + window])

#     df = pd.DataFrame({"realized": real_list,
#                        "garch":    g_list,
#                        "egarch":   e_list},
#                       index=dates).dropna()

#     d_g = df["realized"] - df["garch"]
#     d_e = df["realized"] - df["egarch"]

#     rmse_g = np.sqrt(np.mean(d_g**2))
#     rmse_e = np.sqrt(np.mean(d_e**2))
#     mae_g  = np.mean(np.abs(d_g))
#     mae_e  = np.mean(np.abs(d_e))
#     mape_g = np.mean(np.abs(d_g / df["realized"])) * 100
#     mape_e = np.mean(np.abs(d_e / df["realized"])) * 100
#     t_stat, t_p = stats.ttest_rel(d_g**2, d_e**2)

#     metrics = {
#         "rmse_garch": rmse_g, "rmse_egarch": rmse_e,
#         "mae_garch":  mae_g,  "mae_egarch":  mae_e,
#         "mape_garch": mape_g, "mape_egarch": mape_e,
#         "t_p": t_p,
#     }
#     return metrics, df

# print("\n=== ROLLING OOS FORECASTS (SPX & Momentum) ===")
# oos = {}
# for col in ["SPX", "Momentum"]:
#     print(f"  Rolling OOS: {col} (this takes ~2 mins) ...")
#     oos[col] = {}
#     oos[col]["metrics"], oos[col]["df"] = rolling_oos(returns[col])

# # OOS Results Table
# oos_rows = []
# for col in ["SPX", "Momentum"]:
#     m = oos[col]["metrics"]
#     oos_rows.append({
#         "Factor":          col,
#         "GARCH_RMSE":      round(m["rmse_garch"], 5),
#         "EGARCH_RMSE":     round(m["rmse_egarch"], 5),
#         "RMSE_Imp_%":      round((1 - m["rmse_egarch"] / m["rmse_garch"]) * 100, 2),
#         "GARCH_MAE":       round(m["mae_garch"], 5),
#         "EGARCH_MAE":      round(m["mae_egarch"], 5),
#         "MAE_Imp_%":       round((1 - m["mae_egarch"] / m["mae_garch"]) * 100, 2),
#         "T_Pval":          round(m["t_p"], 4),
#         "Significant":     "YES" if m["t_p"] < 0.05 else "NO"
#     })

# oos_df = pd.DataFrame(oos_rows)
# oos_df.to_csv("OutofSampleResults.csv", index=False)
# print("\n=== OOS RESULTS ===")
# print(oos_df.to_string(index=False))

# # =============================================================================
# # SECTION 11: OOS FORECAST PLOT (SPX)
# # =============================================================================
# fig, ax = plt.subplots(figsize=(13, 5))
# df_spx = oos["SPX"]["df"]
# ax.plot(df_spx.index, df_spx["realized"], lw=1.0, color="black",  label="Realized Vol")
# ax.plot(df_spx.index, df_spx["garch"],    lw=1.0, color="steelblue", label="GARCH Forecast")
# ax.plot(df_spx.index, df_spx["egarch"],   lw=1.0, color="crimson",
#         linestyle="--", label="EGARCH Forecast")
# ax.axvspan(pd.Timestamp("2020-02-20"), pd.Timestamp("2020-04-30"),
#            alpha=0.1, color="orange", label="COVID Fever")
# ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2023-06-30"),
#            alpha=0.1, color="purple", label="Rate Hike Cycle")
# ax.set_title("SPX: Rolling OOS Vol Forecasts — GARCH vs EGARCH", fontsize=12)
# ax.set_ylabel("Volatility (σ)")
# ax.legend(fontsize=9)
# plt.tight_layout()
# save_plot(fig, "06_oos_spx.png")

# # =============================================================================
# # SECTION 12: TAIL RISK — 75th PERCENTILE UNDER-PREDICTION
# # =============================================================================
# print("\n=== TAIL RISK UNDER-PREDICTION (>75th pct Realized Vol) ===")
# tail_rows = []
# for col in ["SPX", "Momentum"]:
#     df = oos[col]["df"]
#     thresh = df["realized"].quantile(0.75)
#     high   = df[df["realized"] >= thresh]

#     g_under = (high["realized"] - high["garch"]).mean()
#     e_under = (high["realized"] - high["egarch"]).mean()
#     imp_pct = (g_under - e_under) / abs(g_under) * 100 if g_under != 0 else np.nan

#     print(f"  {col}: GARCH under {g_under:.4f} | EGARCH under {e_under:.4f} | "
#           f"Improvement {imp_pct:.1f}%")
#     tail_rows.append({
#         "Factor": col, "GARCH_Underpred": g_under,
#         "EGARCH_Underpred": e_under, "Improvement_%": imp_pct
#     })

# tail_df = pd.DataFrame(tail_rows)
# tail_df.to_csv("TailRiskImprovement.csv", index=False)

# # =============================================================================
# # SECTION 13: ACCURACY BY REGIME (Extended from COVID to all periods)
# # =============================================================================
# def regime_accuracy(df_oos: pd.DataFrame) -> pd.DataFrame:
#     """GARCH vs EGARCH RMSE broken down by market regime."""
#     def label(date):
#         if date < pd.Timestamp("2017-01-01"):   return "Pre-2017"
#         if date < pd.Timestamp("2020-01-01"):   return "Bull 2017-19"
#         if date < pd.Timestamp("2020-04-16"):   return "COVID Crisis"
#         if date < pd.Timestamp("2022-01-01"):   return "Recovery 20-21"
#         if date < pd.Timestamp("2024-01-01"):   return "Rate Hike 22-23"
#         return "Post-Hike 24+"

#     df = df_oos.copy()
#     df["regime"] = df.index.map(label)
#     df["g_err2"] = (df["realized"] - df["garch"])**2
#     df["e_err2"] = (df["realized"] - df["egarch"])**2

#     out = df.groupby("regime").agg(
#         GARCH_RMSE =("g_err2", lambda x: np.sqrt(x.mean())),
#         EGARCH_RMSE=("e_err2", lambda x: np.sqrt(x.mean())),
#         N          =("realized", "count")
#     )
#     out["EGARCH_Advantage_%"] = (out["GARCH_RMSE"] - out["EGARCH_RMSE"]) / out["GARCH_RMSE"] * 100
#     return out.round(5)

# print("\n=== REGIME-BASED FORECAST ACCURACY ===")
# for col in ["SPX", "Momentum"]:
#     print(f"\n  {col}:")
#     reg = regime_accuracy(oos[col]["df"])
#     print(reg.to_string())
#     reg.to_csv(f"RegimeAccuracy_{col}.csv")

# # =============================================================================
# # SECTION 14: DIRECTIONAL ACCURACY
# # =============================================================================
# print("\n=== DIRECTIONAL ACCURACY ===")
# dir_rows = []
# for col in ["SPX", "Momentum"]:
#     df = oos[col]["df"].copy()
#     real_d = np.sign(df["realized"].diff()).dropna()
#     g_d    = np.sign(df["garch"].diff()).dropna()
#     e_d    = np.sign(df["egarch"].diff()).dropna()
#     idx    = real_d.index.intersection(g_d.index).intersection(e_d.index)

#     g_acc  = (g_d.loc[idx] == real_d.loc[idx]).mean() * 100
#     e_acc  = (e_d.loc[idx] == real_d.loc[idx]).mean() * 100
#     e_only = ((g_d.loc[idx] != real_d.loc[idx]) & (e_d.loc[idx] == real_d.loc[idx])).sum()

#     dir_rows.append({"Factor": col, "GARCH_Dir_%": round(g_acc,2),
#                      "EGARCH_Dir_%": round(e_acc,2), "EGARCH_Unique_Wins": int(e_only)})
#     print(f"  {col}: GARCH {g_acc:.1f}% | EGARCH {e_acc:.1f}% | EGARCH unique wins: {e_only}")

# pd.DataFrame(dir_rows).to_csv("DirectionalAccuracy.csv", index=False)

# # =============================================================================
# # SECTION 15: FINAL SUMMARY PRINTOUT
# # =============================================================================
# print("\n" + "="*60)
# print("KEY FINDINGS SUMMARY")
# print("="*60)
# for col in factor_cols:
#     res = all_results[col]
#     winner = "EGARCH" if res["aic_diff"] < 0 else "GARCH"
#     print(f"  {col:<12} gamma={res['e_gamma']:+.4f}  "
#           f"persist={res['g_persist']:.4f}  "
#           f"ΔAIC={res['aic_diff']:+.1f}  →  {winner}")

# print("\nOutputs: CompleteAnalysisComparison.csv | OutofSampleResults.csv")
# print("         TailRiskImprovement.csv | DirectionalAccuracy.csv")
# print("         RegimeAccuracy_SPX.csv  | RegimeAccuracy_Momentum.csv")
# print("Plots:   01_prices.png → 06_oos_spx.png")
# print("="*60)